## 🎯 Learning Objectives
* Understand the necessity and benefits of developing custom tools for CrewAI agents.
* Learn how to create a custom tool using CrewAI's `BaseTool` and Pydantic for input validation.
* Implement and configure tool caching within CrewAI to optimize performance and reduce costs.
* Analyze the impact of tool caching on agent execution time and resource utilization.
* Identify appropriate use cases for custom tools and tool caching in advanced AI agent systems.


## Custom Tool Development and Tool Caching in CrewAI

In the realm of advanced AI agents, the ability to interact with the real world, internal systems, and specialized data sources is paramount. While CrewAI provides a robust set of built-in tools, real-world business automation often demands highly specialized functionalities that are unique to an organization's infrastructure or specific workflows. This is where **custom tool development** becomes indispensable.

### The Need for Custom Tools: Extending Agent Capabilities

Imagine an expert financial analyst agent whose task is to generate a quarterly performance report. While it can use general web search tools, it also needs to access a proprietary internal database, run a specific financial modeling algorithm, or interact with a legacy ERP system. These are not off-the-shelf capabilities. Just as a master craftsman needs custom-forged tools for intricate tasks that no mass-produced wrench can handle, your AI agents require custom tools to perform highly specialized, domain-specific actions.

**What is a Custom Tool?**

In CrewAI, a custom tool is essentially a Python class that encapsulates a specific action or capability. It inherits from `crewai.tools.BaseTool` and defines:

1.  **`name`**: A unique, descriptive name for the tool.
2.  **`description`**: A clear explanation of what the tool does, its purpose, and how an agent should use it. This is crucial for the LLM to understand when to invoke the tool.
3.  **`args_schema` (Optional but Recommended)**: A Pydantic `BaseModel` that defines the expected input parameters for the tool. This ensures type safety and provides a structured way for the LLM to call the tool with correct arguments.
4.  **`_run` method**: The core logic of the tool, containing the actual Python code that performs the desired action. This method receives the validated arguments defined in `args_schema`.

By developing custom tools, you empower your agents to interact with virtually any system or data source, transforming them from general-purpose assistants into highly specialized, integrated components of your business processes.

### Optimizing Performance with Tool Caching

As agents become more sophisticated, they might invoke tools frequently, especially if a task requires repetitive data retrieval or computation. Many tool calls, particularly those interacting with external APIs or performing complex calculations, can be expensive in terms of time, computational resources, or even monetary cost (e.g., paid API calls).

This is where **tool caching** comes into play. Caching is a technique where the results of expensive operations are stored (cached) after their first execution. If the same operation is requested again with the same inputs, the cached result is returned immediately instead of re-executing the operation. This significantly reduces latency, saves resources, and improves the overall efficiency of your agentic system.

**How Tool Caching Works (Conceptually):**

1.  An agent decides to use a tool with specific input parameters.
2.  Before executing the tool's `_run` method, the system checks if a result for these exact inputs already exists in the cache.
3.  If a cached result is found, it's returned instantly.
4.  If no cached result is found, the tool's `_run` method is executed.
5.  The result of the execution is then stored in the cache, associated with the input parameters, before being returned to the agent.

**When to use Caching:**

*   **Expensive API calls:** Reduce calls to paid services or services with rate limits.
*   **Slow computations:** Avoid re-running time-consuming algorithms.
*   **Idempotent operations:** Tools that produce the same output for the same input and have no side effects.
*   **Data that doesn't change frequently:** If the data retrieved by a tool is relatively static, caching is highly effective.

**When to be cautious with Caching:**

*   **Real-time data:** If a tool fetches data that changes constantly and freshness is critical, caching might provide stale information.
*   **Tools with side effects:** If a tool modifies external state (e.g., writes to a database, sends an email), caching its result would prevent these side effects from occurring on subsequent calls.

CrewAI provides built-in mechanisms to enable and manage tool caching, allowing you to fine-tune your agents' performance without complex manual implementations. By strategically combining custom tools with intelligent caching, you can build highly efficient, robust, and cost-effective AI automation solutions.


In [ ]:
import os
import time
from typing import Type

from pydantic import BaseModel, Field
from crewai import Agent, Task, Crew, Process
from crewai_tools import BaseTool

# --- 2026 Ready: Mock LLM Setup for demonstration ---
# In a real 2026 scenario, you'd likely use a local LLM (e.g., Ollama, vLLM) or a specific cloud provider.
# For this example, we'll use a placeholder that mimics an LLM response.
class MockLLM:
    def __init__(self, model_name="mock-model-2026"):
        self.model_name = model_name

    def invoke(self, prompt):
        # Simulate LLM processing time and generate a generic response
        time.sleep(0.5) 
        return f"Mock LLM response for prompt: {prompt[:50]}..."

    def chat(self, messages):
        # Simulate chat interaction
        time.sleep(0.5)
        last_message = messages[-1]['content'] if messages else ""
        return f"Mock LLM chat response for: {last_message[:50]}..."

    def __call__(self, prompt):
        return self.invoke(prompt)

mock_llm = MockLLM()

# --- 1. Custom Tool Definition ---

# Define the input schema for our custom tool using Pydantic
class ProprietaryDataAnalyzerInput(BaseModel):
    query: str = Field(description="The specific query or request for proprietary data analysis.")
    report_type: str = Field(description="The type of report to generate (e.g., 'financial', 'market_trend', 'operational').")

class ProprietaryDataAnalyzerTool(BaseTool):
    name: str = "Proprietary Data Analyzer"
    description: str = (
        "Analyzes proprietary internal business data based on a given query and report type. "
        "This tool simulates accessing a specialized, slow internal data warehouse. "
        "Expect a delay in results due to complex data processing."
    )
    args_schema: Type[BaseModel] = ProprietaryDataAnalyzerInput

    def _run(self, query: str, report_type: str) -> str:
        """Simulates a complex, time-consuming proprietary data analysis."""
        print(f"\n[ProprietaryDataAnalyzerTool]: Executing analysis for query: '{query}' (Type: {report_type})... This will take a moment.")
        # Simulate a significant delay to highlight caching benefits
        time.sleep(3) 
        
        # Simulate different results based on query/report_type for realism
        if "financial" in report_type.lower() and "Q3" in query:
            result = "Proprietary Financial Report (Q3 2025): Revenue up 12%, Profit Margin 8.5%. Key growth areas: AI Services."
        elif "market_trend" in report_type.lower() and "AI" in query:
            result = "Proprietary Market Trend Analysis (AI Sector): Significant growth expected in Agentic AI tools, 25% CAGR over next 3 years."
        else:
            result = f"Proprietary Data Analysis for '{query}' ({report_type}): No specific report found, but general trends indicate stable performance."
            
        print(f"[ProprietaryDataAnalyzerTool]: Analysis complete for query: '{query}'.")
        return result

# Instantiate our custom tool
proprietary_analyzer_tool = ProprietaryDataAnalyzerTool()

# --- 2. Agent Definition ---

# Define an agent that can use our custom tool
research_analyst = Agent(
    role='Senior Research Analyst',
    goal='Conduct in-depth analysis of internal proprietary data to inform strategic decisions.',
    backstory=(
        "You are a highly experienced Senior Research Analyst with access to exclusive internal data sources. "
        "Your expertise lies in extracting critical insights from complex datasets and presenting them clearly." 
        "You are meticulous and always verify information."
    ),
    tools=[proprietary_analyzer_tool], # Integrate the custom tool
    llm=mock_llm, # Use our mock LLM
    verbose=True,
    allow_delegation=False
)

# --- 3. Task Definition ---

# Define a task that requires the custom tool
analysis_task = Task(
    description=(
        "Analyze the company's Q3 2025 financial performance using proprietary data. "
        "Focus on revenue growth, profit margins, and identify key contributing factors. "
        "Generate a concise summary report of the findings." 
        "Use the 'Proprietary Data Analyzer' tool with 'financial' report_type and 'Q3 2025 financial performance' query." 
        "After analysis, summarize the key insights."
    ),
    agent=research_analyst,
    expected_output="A concise summary report (2-3 paragraphs) detailing Q3 2025 financial performance, including revenue growth, profit margins, and key contributing factors, based on proprietary data analysis."
)

# --- 4. Crew Definition and Execution (Without Caching First) ---

print("\n--- Running Crew WITHOUT Caching (First Run) ---")
crew_no_cache_first_run = Crew(
    agents=[research_analyst],
    tasks=[analysis_task],
    process=Process.sequential,
    llm=mock_llm, # Ensure LLM is passed to Crew
    verbose=True,
    cache=False # Explicitly disable caching for comparison
)

start_time_no_cache_first = time.time()
result_no_cache_first = crew_no_cache_first_run.kickoff()
end_time_no_cache_first = time.time()
print(f"\nCrew WITHOUT Caching (First Run) took: {end_time_no_cache_first - start_time_no_cache_first:.2f} seconds")
print("\n--- Result WITHOUT Caching (First Run) ---")
print(result_no_cache_first)

print("\n--- Running Crew WITHOUT Caching (Second Run - Same Task) ---")
# Re-instantiate crew to ensure a fresh run if needed, though cache=False should prevent caching anyway
crew_no_cache_second_run = Crew(
    agents=[research_analyst],
    tasks=[analysis_task],
    process=Process.sequential,
    llm=mock_llm,
    verbose=True,
    cache=False
)
start_time_no_cache_second = time.time()
result_no_cache_second = crew_no_cache_second_run.kickoff()
end_time_no_cache_second = time.time()
print(f"\nCrew WITHOUT Caching (Second Run) took: {end_time_no_cache_second - start_time_no_cache_second:.2f} seconds")
print("\n--- Result WITHOUT Caching (Second Run) ---")
print(result_no_cache_second)


# --- 5. Crew Definition and Execution (WITH Caching) ---

print("\n\n--- Running Crew WITH Caching (First Run) ---")
crew_with_cache_first_run = Crew(
    agents=[research_analyst],
    tasks=[analysis_task],
    process=Process.sequential,
    llm=mock_llm,
    verbose=True,
    cache=True # Enable caching
)

start_time_with_cache_first = time.time()
result_with_cache_first = crew_with_cache_first_run.kickoff()
end_time_with_cache_first = time.time()
print(f"\nCrew WITH Caching (First Run) took: {end_time_with_cache_first - start_time_with_cache_first:.2f} seconds")
print("\n--- Result WITH Caching (First Run) ---")
print(result_with_cache_first)

print("\n--- Running Crew WITH Caching (Second Run - Same Task) ---")
# For caching to be effective, the Crew instance should ideally persist or share a cache.
# CrewAI's default caching is often session-based or file-based depending on configuration.
# For this demo, re-running the same crew with cache=True will demonstrate the effect.
crew_with_cache_second_run = Crew(
    agents=[research_analyst],
    tasks=[analysis_task],
    process=Process.sequential,
    llm=mock_llm,
    verbose=True,
    cache=True # Caching is still enabled
)
start_time_with_cache_second = time.time()
result_with_cache_second = crew_with_cache_second_run.kickoff()
end_time_with_cache_second = time.time()
print(f"\nCrew WITH Caching (Second Run) took: {end_time_with_cache_second - start_time_with_cache_second:.2f} seconds")
print("\n--- Result WITH Caching (Second Run) ---")
print(result_with_cache_second)

print("\n--- Running Crew WITH Caching (Third Run - Different Task to show cache miss) ---")
# Define a new task with different parameters to demonstrate a cache miss
market_analysis_task = Task(
    description=(
        "Analyze the current market trends in the AI sector using proprietary data. "
        "Identify key growth areas and potential challenges. "
        "Generate a concise summary report of the findings." 
        "Use the 'Proprietary Data Analyzer' tool with 'market_trend' report_type and 'AI sector growth' query." 
        "After analysis, summarize the key insights."
    ),
    agent=research_analyst,
    expected_output="A concise summary report (2-3 paragraphs) detailing current AI sector market trends, including growth areas and challenges, based on proprietary data analysis."
)

crew_with_cache_third_run = Crew(
    agents=[research_analyst],
    tasks=[market_analysis_task],
    process=Process.sequential,
    llm=mock_llm,
    verbose=True,
    cache=True
)
start_time_with_cache_third = time.time()
result_with_cache_third = crew_with_cache_third_run.kickoff()
end_time_with_cache_third = time.time()
print(f"\nCrew WITH Caching (Third Run - Different Task) took: {end_time_with_cache_third - start_time_with_cache_third:.2f} seconds")
print("\n--- Result WITH Caching (Third Run - Different Task) ---")
print(result_with_cache_third)


### Interpreting the Code Output and Performance Trade-offs

When you run the provided code, you will observe distinct differences in execution time, particularly between the runs with and without caching enabled.

**Expected Output Interpretation:**

1.  **Runs WITHOUT Caching (First and Second):**
    *   You will see the `[ProprietaryDataAnalyzerTool]: Executing analysis...` message printed for *both* the first and second runs. This indicates that the `_run` method of our custom tool is being executed each time, including the simulated `time.sleep(3)` delay.
    *   The reported execution time for both runs will be approximately the same (around 4-5 seconds, depending on LLM mock time and other overhead), reflecting the full execution of the tool each time.
    *   This demonstrates that without caching, every identical tool call incurs its full cost (time, API calls, computation).

2.  **Runs WITH Caching (First Run):**
    *   The first run with caching enabled will behave similarly to the non-cached runs. You will see the `[ProprietaryDataAnalyzerTool]: Executing analysis...` message, and the execution time will be similar (around 4-5 seconds).
    *   This is because the cache is initially empty. The tool executes, and its result (associated with the specific `query` and `report_type`) is stored in the cache.

3.  **Runs WITH Caching (Second Run - Same Task):**
    *   This is where caching's benefit becomes evident. You will *not* see the `[ProprietaryDataAnalyzerTool]: Executing analysis...` message. Instead, CrewAI will internally recognize that the agent is attempting to call the `Proprietary Data Analyzer` tool with the *exact same inputs* as before.
    *   The system will retrieve the result directly from the cache. The reported execution time will be significantly shorter (likely less than 1 second), as the expensive `time.sleep(3)` operation is skipped entirely.
    *   This clearly illustrates how caching bypasses re-execution of expensive tool calls, leading to substantial time savings.

4.  **Runs WITH Caching (Third Run - Different Task):**
    *   For this run, the agent is asked to perform a *different* analysis (`'AI sector growth'` and `'market_trend'`). Since these inputs are different from the previous cached call, the cache will *miss*.
    *   You will again see the `[ProprietaryDataAnalyzerTool]: Executing analysis...` message, and the execution time will be similar to the initial non-cached runs (around 4-5 seconds).
    *   This demonstrates that caching is input-specific. Only identical calls benefit from the cache; new inputs trigger a fresh execution and a new cache entry.

**Performance Trade-offs:**

*   **Speed vs. Freshness:** Caching dramatically improves speed and reduces resource consumption. However, it introduces a trade-off with data freshness. If the underlying data accessed by your custom tool changes frequently, a cached result might become stale. For such scenarios, consider implementing cache invalidation strategies or setting appropriate Time-To-Live (TTL) for cache entries.
*   **Memory/Storage vs. Computation:** Caching requires memory or disk space to store results. For tools that return very large outputs or are called with an extremely wide variety of inputs, the cache size could grow significantly. This is generally a minor concern for most agentic applications but worth considering for high-scale systems.
*   **Complexity:** While CrewAI simplifies caching, custom cache implementations (e.g., using Redis for distributed caching) can add complexity. For most use cases, CrewAI's built-in caching is sufficient.

**Typical Use Cases:**

*   **Custom Tools:**
    *   **Internal API Integration:** Tools to interact with your company's proprietary microservices, CRM, ERP, or data warehouses.
    *   **Specialized Data Processing:** Tools that perform complex data transformations, machine learning model inferences, or statistical analyses using internal libraries.
    *   **Legacy System Interaction:** Tools to bridge modern AI agents with older, non-standard systems.
    *   **Domain-Specific Actions:** Tools for unique actions like generating specific reports, managing internal project tickets, or triggering custom deployment pipelines.

*   **Tool Caching:**
    *   **Expensive LLM API Calls:** If a tool internally makes calls to another LLM (e.g., for summarization or classification), caching these results can save significant costs.
    *   **External Data Fetching:** Tools that query external databases, financial APIs, or web services where calls are rate-limited or billed per request.
    *   **Complex Computations:** Tools that run computationally intensive algorithms (e.g., simulations, optimizations) where the same inputs are likely to be encountered multiple times.
    *   **Static Reference Data:** Tools that retrieve configuration settings, lookup tables, or relatively static reference data that doesn't change often.


### Resources

*   **CrewAI Documentation - Tools:** Explore the official documentation for creating and managing tools in CrewAI.
    *   [https://docs.crewai.com/how-to/create-custom-tools/](https://docs.crewai.com/how-to/create-custom-tools/)
*   **CrewAI Documentation - Caching:** Understand how to configure and leverage caching in your CrewAI projects.
    *   [https://docs.crewai.com/how-to/cache-results/](https://docs.crewai.com/how-to/cache-results/)
*   **Pydantic Documentation:** Learn more about Pydantic for robust data validation and settings management, essential for defining tool input schemas.
    *   [https://docs.pydantic.dev/latest/](https://docs.pydantic.dev/latest/)
*   **Designing Robust Tools for AI Agents:** An article discussing best practices for tool design in agentic systems (conceptual, not specific to CrewAI).
    *   [Search for "Designing Tools for LLM Agents" on Google Scholar or Medium for various perspectives and best practices as of 2026.]
